In [ ]:
!pip install pysam

In [ ]:
import gzip
import numpy as np
import pandas as pd
import statistics
from tqdm import tqdm
import matplotlib.pyplot as plt
import pysam
import pyspark
import dxpy
import dxdata
import sys

In [ ]:
!dx download 'Bulk/Exome sequences/Exome OQFE CRAM files/helper_files/GRCh38_full_analysis_set_plus_decoy_hla.fa'
!dx download 'Bulk/Exome sequences/Exome OQFE CRAM files/helper_files/GRCh38_full_analysis_set_plus_decoy_hla.fa.fai'
fasta = pysam.FastaFile('GRCh38_full_analysis_set_plus_decoy_hla.fa')

In [ ]:
lengths = dict(zip(fasta.references, fasta.lengths))
lengths = {chrom: lengths[chrom] for chrom in lengths 
           if chrom[3:] in [str(i) for i in range(1, 23)]}

In [ ]:
def plot_ibd(ibds, ibd):
    
    cnts = []

    for pair in ibds:
        cnt = sum([ibds[pair][ibd][chrom]['Length'] 
                   for chrom in range(1, 23)])
        cnts.append(cnt/sum(lengths.values()))

    print(round(min(cnts), 4), round(max(cnts), 4), round(statistics.mean(cnts), 4), 
          round(statistics.median(cnts), 4), len(cnts))

    plt.hist(cnts, bins = 20)
    plt.xlabel(f'Fraction of the genome shared {ibd}')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()
    plt.clf()

# Read data

In [ ]:
sibs = {}

with open('/mnt/project/DNM/sibs_pairs.txt') as f:
    
    for row in f:
        
        row = row.strip()
        row = row.split(' ')
        
        sib1 = str(row[0])
        sib2 = str(row[1])
        
        sibs[(sib1, sib2)] = True

In [ ]:
sibs_ibds = {}

for chrom in tqdm(range(1, 23)):

    with gzip.open(f'/mnt/project/DNM/snipar/IBD/chr{chrom}.ibd.segments.gz', 'r') as f:

        next(f)

        for row in f:

            row = row.strip()
            row = row.split()

            ind1 = str(min(int(row[0]), int(row[1])))
            ind2 = str(max(int(row[0]), int(row[1])))

            if (ind1, ind2) not in sibs:
                continue

            ibd = int(row[2])

            if (ind1, ind2) not in sibs_ibds:
                sibs_ibds[(ind1, ind2)] = {}
                sibs_ibds[(ind1, ind2)]['IBD0'] = {c: {'Ranges': [], 'Length': 0} for c in range(1, 23)}
                sibs_ibds[(ind1, ind2)]['IBD1'] = {c: {'Ranges': [], 'Length': 0} for c in range(1, 23)}
                sibs_ibds[(ind1, ind2)]['IBD2'] = {c: {'Ranges': [], 'Length': 0} for c in range(1, 23)}

            start = int(row[4])
            end = int(row[5])

            sibs_ibds[(ind1, ind2)][f'IBD{ibd}'][chrom]['Ranges'].append((start, end))
            sibs_ibds[(ind1, ind2)][f'IBD{ibd}'][chrom]['Length'] += end-start+1

# Complete predictions

In [ ]:
plot_ibd(sibs_ibds, 'IBD0')
plot_ibd(sibs_ibds, 'IBD1')
plot_ibd(sibs_ibds, 'IBD2')

## Record IBD segments

In [ ]:
with open('sibs_ibd0.bed', 'w') as f:
    
    for pair in sibs_ibds:
        
        f.write(f'>start:{pair[0]}-{pair[1]}\n')
        
        idx = 0
        for chrom in range(1, 23):
            for (start, end) in sibs_ibds[pair]['IBD0'][chrom]['Ranges']:
                f.write(f'{chrom} {start} {end+1} {idx}\n')
                idx += 1
                
        f.write(f'>end:{pair[0]}-{pair[1]}\n')

In [ ]:
with open('sibs_ibd1.bed', 'w') as f:
    
    for pair in sibs_ibds:
        
        f.write(f'>start:{pair[0]}-{pair[1]}\n')
        
        idx = 0
        for chrom in range(1, 23):
            for (start, end) in sibs_ibds[pair]['IBD1'][chrom]['Ranges']:
                f.write(f'{chrom} {start} {end+1} {idx}\n')
                idx += 1
                
        f.write(f'>end:{pair[0]}-{pair[1]}\n')

In [ ]:
with open('sibs_ibd2.bed', 'w') as f:
    
    for pair in sibs_ibds:
        
        f.write(f'>start:{pair[0]}-{pair[1]}\n')
        
        idx = 0
        for chrom in range(1, 23):
            for (start, end) in sibs_ibds[pair]['IBD2'][chrom]['Ranges']:
                f.write(f'{chrom} {start} {end+1} {idx}\n')
                idx += 1
                
        f.write(f'>end:{pair[0]}-{pair[1]}\n')

In [ ]:
!dx upload sibs_ibd0.bed --path="DNM/snipar/"
!dx upload sibs_ibd1.bed --path="DNM/snipar/"
!dx upload sibs_ibd2.bed --path="DNM/snipar/"

## Trim ends of segments

In [ ]:
!zcat "/mnt/project/DNM/decode_combined_map.gz" | grep -v "#" > decode_combined_map.gmap

In [ ]:
def trim_ibd2(ibds, trim):
    
    gmap = pd.read_csv('decode_combined_map.gmap', sep = '\t', index_col = None)
    gmap_dict = {chrom: df for chrom, df in gmap.groupby('Chr')}

    ibds_trimmed = {}
    
    for sib1, sib2 in ibds:
        ibds_trimmed[(sib1, sib2)] = {}
        ibds_trimmed[(sib1, sib2)]['IBD2'] = {c: {'Ranges': [], 'Length': 0} for c in range(1, 23)}

    for _, (sib1, sib2) in tqdm(enumerate(ibds)):

        for chrom in ibds[(sib1, sib2)]['IBD2']:

            gmap_chr = gmap_dict[f'chr{chrom}']
            cm_arr = gmap_chr['cM'].values
            bp_arr = gmap_chr['End'].values

            ranges = ibds[(sib1, sib2)]['IBD2'][chrom]['Ranges']

            if not ranges:
                continue

            starts_bp, ends_bp = zip(*ranges)

            starts_cm = np.interp(starts_bp, bp_arr, cm_arr).astype(float)
            ends_cm = np.interp(ends_bp, bp_arr, cm_arr).astype(float)

            new_starts_cm = starts_cm + trim
            new_ends_cm = ends_cm - trim
            mask = new_starts_cm <= new_ends_cm

            new_starts_bp = np.interp(new_starts_cm[mask], cm_arr, bp_arr).astype(int)
            new_ends_bp = np.interp(new_ends_cm[mask], cm_arr, bp_arr).astype(int)

            for i in range(len(new_starts_bp)):
                if new_starts_bp[i] <= new_ends_bp[i]:
                    ibds_trimmed[(sib1, sib2)]['IBD2'][chrom]['Ranges'].append((new_starts_bp[i], new_ends_bp[i]))
                    ibds_trimmed[(sib1, sib2)]['IBD2'][chrom]['Length'] += new_ends_bp[i]-new_starts_bp[i]+1
            
    return ibds_trimmed

In [ ]:
for trim in [0.25, 0.5, 0.75, 1]:
    
    sibs_ibds_trimmed = trim_ibd2(sibs_ibds, trim)
    plot_ibd(sibs_ibds_trimmed, 'IBD2')

    with open(f'sibs_ibd2_{trim}cM_trim.bed', 'w') as f:

        for pair in sibs_ibds_trimmed:

            f.write(f'>start:{pair[0]}-{pair[1]}\n')

            idx = 0
            for chrom in range(1, 23):
                for (start, end) in sibs_ibds_trimmed[pair]['IBD2'][chrom]['Ranges']:
                    f.write(f'{chrom} {start} {end+1} {idx}\n')
                    idx += 1

            f.write(f'>end:{pair[0]}-{pair[1]}\n')

In [ ]:
!dx upload sibs_ibd2_*cM_trim.bed --path="DNM/snipar/"

# Exclude short segments

In [ ]:
def include_segments(ibds, cm_lower, cm_upper):

    gmap = pd.read_csv('decode_combined_map.gmap', sep = '\t', index_col = None)
    gmap_dict = {chrom: df for chrom, df in gmap.groupby('Chr')}
    
    ibds_inc = {}
    
    for sib1, sib2 in ibds:
        ibds_inc[(sib1, sib2)] = {}
        ibds_inc[(sib1, sib2)]['IBD2'] = {c: {'Ranges': [], 'Length': 0} for c in range(1, 23)}

    for _, (sib1, sib2) in tqdm(enumerate(ibds)):
        
        for chrom in ibds[(sib1, sib2)]['IBD2']:

            gmap_chr = gmap_dict[f'chr{chrom}']
            cm_arr = gmap_chr['cM'].values
            bp_arr = gmap_chr['End'].values

            ranges = ibds[(sib1, sib2)]['IBD2'][chrom]['Ranges']

            if not ranges:
                continue

            starts_bp, ends_bp = zip(*ranges)

            starts_cm = np.interp(starts_bp, bp_arr, cm_arr).astype(float)
            ends_cm = np.interp(ends_bp, bp_arr, cm_arr).astype(float)
            
            for i in range(len(starts_bp)):
                if (ends_cm[i]-starts_cm[i]) >= cm_lower and (ends_cm[i]-starts_cm[i]) < cm_upper:
                    ibds_inc[(sib1, sib2)]['IBD2'][chrom]['Ranges'].append((starts_bp[i], ends_bp[i]))
                    ibds_inc[(sib1, sib2)]['IBD2'][chrom]['Length'] += ends_bp[i]-starts_bp[i]+1
            
    return ibds_inc

In [ ]:
def exclude_segments(ibds, cm_lower, cm_upper):
    
    gmap = pd.read_csv('decode_combined_map.gmap', sep = '\t', index_col = None)
    gmap_dict = {chrom: df for chrom, df in gmap.groupby('Chr')}
    
    ibds_exc = {}
    
    for sib1, sib2 in ibds:
        ibds_exc[(sib1, sib2)] = {}
        ibds_exc[(sib1, sib2)]['IBD2'] = {c: {'Ranges': [], 'Length': 0} for c in range(1, 23)}

    for _, (sib1, sib2) in tqdm(enumerate(ibds)):
        
        for chrom in ibds[(sib1, sib2)]['IBD2']:
            
            gmap_chr = gmap_dict[f'chr{chrom}']
            cm_arr = gmap_chr['cM'].values
            bp_arr = gmap_chr['End'].values

            ranges = ibds[(sib1, sib2)]['IBD2'][chrom]['Ranges']

            if not ranges:
                continue

            starts_bp, ends_bp = zip(*ranges)

            starts_cm = np.interp(starts_bp, bp_arr, cm_arr).astype(float)
            ends_cm = np.interp(ends_bp, bp_arr, cm_arr).astype(float)

            for i in range(len(starts_bp)):
                if (ends_cm[i]-starts_cm[i]) < cm_lower or (ends_cm[i]-starts_cm[i]) >= cm_upper:
                    ibds_exc[(sib1, sib2)]['IBD2'][chrom]['Ranges'].append((starts_bp[i], ends_bp[i]))
                    ibds_exc[(sib1, sib2)]['IBD2'][chrom]['Length'] += ends_bp[i]-starts_bp[i]+1
            
    return ibds_exc

In [ ]:
for trim in [0.75]:
    
    for cm_lower, cm_upper in [(0, 2), (2, 4), (4, 6), (6, 10), 
                               (10, 20), (20, sys.maxsize/(10**6))]:
    
        sibs_ibds_trimmed = trim_ibd2(sibs_ibds, trim)
        sibs_ibds_inc = include_segments(sibs_ibds_trimmed, cm_lower, cm_upper)
        plot_ibd(sibs_ibds_inc, 'IBD2')

        with open(f'sibs_ibd2_{trim}cM_trim_{cm_lower}cM_{cm_upper}cM_inc.bed', 'w') as f:

            for pair in sibs_ibds_inc:

                f.write(f'>start:{pair[0]}-{pair[1]}\n')

                idx = 0
                for chrom in range(1, 23):
                    for (start, end) in sibs_ibds_inc[pair]['IBD2'][chrom]['Ranges']:
                        f.write(f'{chrom} {start} {end+1} {idx}\n')
                        idx += 1

                f.write(f'>end:{pair[0]}-{pair[1]}\n')

In [ ]:
for trim in [0.75]:
    
    for cm_lower, cm_upper in [(0, 4), (0, 6)]:
    
        sibs_ibds_trimmed = trim_ibd2(sibs_ibds, trim)
        sibs_ibds_exc = exclude_segments(sibs_ibds_trimmed, cm_lower, cm_upper)
        plot_ibd(sibs_ibds_exc, 'IBD2')

        with open(f'sibs_ibd2_{trim}cM_trim_{cm_lower}cM_{cm_upper}cM_exc.bed', 'w') as f:

            for pair in sibs_ibds_exc:

                f.write(f'>start:{pair[0]}-{pair[1]}\n')

                idx = 0
                for chrom in range(1, 23):
                    for (start, end) in sibs_ibds_exc[pair]['IBD2'][chrom]['Ranges']:
                        f.write(f'{chrom} {start} {end+1} {idx}\n')
                        idx += 1

                f.write(f'>end:{pair[0]}-{pair[1]}\n')

In [ ]:
!dx upload sibs_ibd2_*cM_trim_*cM_*cM_inc.bed --path="DNM/snipar/"
!dx upload sibs_ibd2_*cM_trim_*cM_*cM_exc.bed --path="DNM/snipar/"